# 10 · Temporal Convolutional Network and the ensemble

A small TCN trained from scratch to carry the weather signal: 56 days of demand through five dilated causal residual blocks (dilations 1–16, 32 channels), plus a 7×10 future block (3 normalised GFS temperatures, 4 calendar terms, 3 availability masks). Huber loss, AdamW, early stopping on 2023.

**Key results**
- TCN 2024 MAE 7.27 MU (best epoch 16 of 36).
- Errors are complementary: Chronos-2 over-predicts (+1.10 MU bias), TCN under-predicts (−4.75).
- **50/50 ensemble: 6.05 MU MAE, 8.12 RMSE** on 2024. Weights fixed here and frozen before any 2025–26 data was touched.

**Outputs:** `models/gridpulse_ensemble_v1/` (checkpoint with normalisation stats, metadata, 2024 predictions)

> Paths are relative to `notebooks/`. Outputs are from the original research run; see [`data/README.md`](../data/README.md) for data that must be downloaded.

➡ Next: [11 · External validation (2025–26), packaging and deployment checks](11_external_validation_and_packaging.ipynb)

Imports and datasets

In [1]:
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sklearn.metrics import mean_absolute_error

DATA = Path("../data/processed")

demand = pd.read_csv(
    DATA / "gridpulse_v2_development.csv",
    parse_dates=["date"]
).sort_values("date")

gfs = pd.read_parquet(
    DATA / "gridpulse_gfs_previous_runs_2021_2024.parquet"
)

gfs["time"] = pd.to_datetime(gfs["time"])

benchmark = pd.read_parquet(
    DATA / "gridpulse_chronos2_validation_2024.parquet"
)

benchmark["origin"] = pd.to_datetime(benchmark["origin"])
benchmark["target_date"] = pd.to_datetime(benchmark["target_date"])

print("PyTorch:", torch.__version__)
print("Demand:", demand.shape)
print("GFS:", gfs.shape)
print("Benchmark:", benchmark.shape)

PyTorch: 2.15.0+cpu
Demand: (2193, 4)
GFS: (32904, 8)
Benchmark: (2520, 7)


Daily GFS features

In [2]:
lead_cols = [
    f"temperature_2m_previous_day{i}"
    for i in range(1, 8)
]

daily_gfs = (
    gfs.assign(date=gfs["time"].dt.normalize())
    .groupby("date")[lead_cols]
    .agg(["mean", "max", "min", "count"])
)

daily_gfs.columns = [
    f"{lead}_{stat}"
    for lead, stat in daily_gfs.columns
]

# An incomplete day is not a valid daily temperature forecast.
for lead in lead_cols:
    incomplete = daily_gfs[f"{lead}_count"] != 24
    for stat in ["mean", "max", "min"]:
        daily_gfs.loc[incomplete, f"{lead}_{stat}"] = np.nan

print("Daily GFS records:", len(daily_gfs))
print("Days with all leads complete:",
      daily_gfs[[f"{x}_count" for x in lead_cols]]
      .eq(24).all(axis=1).sum())

Daily GFS records: 1371
Days with all leads complete: 1344


Create Raw sequences

In [3]:
demand_series = demand.set_index("date")["energy_met_mu"].sort_index()

def make_sample(origin):
    past_dates = pd.date_range(end=origin, periods=56)
    future_dates = pd.date_range(
        start=origin + pd.Timedelta(days=1),
        periods=7
    )

    past = demand_series.reindex(past_dates).to_numpy(dtype=np.float32)
    target = demand_series.reindex(future_dates).to_numpy(dtype=np.float32)

    weather = []

    for horizon, date in enumerate(future_dates, start=1):
        lead = f"temperature_2m_previous_day{horizon}"

        if date in daily_gfs.index:
            row = daily_gfs.loc[date]
            temp = [
                row[f"{lead}_mean"],
                row[f"{lead}_max"],
                row[f"{lead}_min"]
            ]
        else:
            temp = [np.nan] * 3

        doy = date.dayofyear
        dow = date.dayofweek

        weather.append([
            *temp,
            np.sin(2 * np.pi * doy / 365.25),
            np.cos(2 * np.pi * doy / 365.25),
            np.sin(2 * np.pi * dow / 7),
            np.cos(2 * np.pi * dow / 7),
        ])

    return past, np.asarray(weather, dtype=np.float32), target


# Training: all seven targets strictly before 2024
train_origins = pd.date_range("2021-04-01", "2023-12-24")

# Validation: exactly the frozen benchmark origins
val_origins = pd.DatetimeIndex(
    sorted(benchmark["origin"].unique())
)

print("Candidate training origins:", len(train_origins))
print("Validation origins:", len(val_origins))

sample = make_sample(pd.Timestamp("2021-04-24"))

print("History shape:", sample[0].shape)
print("Future features shape:", sample[1].shape)
print("Target shape:", sample[2].shape)

Candidate training origins: 998
Validation origins: 360
History shape: (56,)
Future features shape: (7, 7)
Target shape: (7,)


Keep valid training and validation windows

In [4]:
def build_arrays(origins):
    histories, futures, targets, kept = [], [], [], []

    for origin in origins:
        past, future, target = make_sample(origin)

        if not np.isfinite(past).all():
            continue

        if not np.isfinite(target).all():
            continue

        histories.append(past)
        futures.append(future)
        targets.append(target)
        kept.append(origin)

    return (
        np.stack(histories),
        np.stack(futures),
        np.stack(targets),
        pd.DatetimeIndex(kept),
    )

Xh_train, Xf_train, y_train, kept_train = build_arrays(train_origins)
Xh_val, Xf_val, y_val, kept_val = build_arrays(val_origins)

print("Training windows:", len(kept_train))
print("Validation windows:", len(kept_val))

print("Training shapes:")
print(Xh_train.shape, Xf_train.shape, y_train.shape)

print("Missing future temperatures:")
print("Train:", np.isnan(Xf_train[:, :, :3]).sum())
print("Validation:", np.isnan(Xf_val[:, :, :3]).sum())

assert len(kept_val) == 360

Training windows: 640
Validation windows: 360
Training shapes:
(640, 56) (640, 7, 7) (640, 7)
Missing future temperatures:
Train: 9
Validation: 399


Normalize demand and temperature

In [5]:
# Demand normalization: training data only
demand_mean = Xh_train.mean()
demand_std = Xh_train.std() + 1e-6

Xh_train_n = (Xh_train - demand_mean) / demand_std
Xh_val_n = (Xh_val - demand_mean) / demand_std

y_train_n = (y_train - demand_mean) / demand_std
y_val_n = (y_val - demand_mean) / demand_std

# Temperature normalization: ignore missing values
temp_mean = np.nanmean(Xf_train[:, :, :3], axis=(0, 1))
temp_std = np.nanstd(Xf_train[:, :, :3], axis=(0, 1)) + 1e-6

print("Demand mean/std:", round(float(demand_mean), 2),
      round(float(demand_std), 2))
print("Temperature mean:", temp_mean.round(2))
print("Temperature std:", temp_std.round(2))

Demand mean/std: 93.25 25.11
Temperature mean: [30.17 34.68 25.98]
Temperature std: [6.8  6.85 6.7 ]


Build normalized future features with masks

In [6]:
def normalize_future(X):
    temp = X[:, :, :3]

    mask = np.isfinite(temp).astype(np.float32)

    normalized = (temp - temp_mean) / temp_std
    normalized = np.where(
        mask.astype(bool), normalized, 0.0
    ).astype(np.float32)

    calendar = X[:, :, 3:].astype(np.float32)

    return np.concatenate(
        [normalized, calendar, mask],
        axis=2
    )

Xf_train_n = normalize_future(Xf_train)
Xf_val_n = normalize_future(Xf_val)

print("Training future shape:", Xf_train_n.shape)
print("Validation future shape:", Xf_val_n.shape)

assert Xf_train_n.shape == (640, 7, 10)
assert Xf_val_n.shape == (360, 7, 10)

assert np.isfinite(Xf_train_n).all()
assert np.isfinite(Xf_val_n).all()

print("Normalization complete.")

Training future shape: (640, 7, 10)
Validation future shape: (360, 7, 10)
Normalization complete.


Temporal convolutional network

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TemporalBlock(nn.Module):
    def __init__(self, channels, dilation):
        super().__init__()

        self.conv = nn.Conv1d(
            channels, channels,
            kernel_size=3,
            dilation=dilation,
            padding=2 * dilation
        )
        self.norm = nn.GroupNorm(4, channels)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        residual = x
        x = self.conv(x)

        # Remove right padding: causal convolution
        x = x[:, :, :-self.conv.padding[0]]

        x = F.relu(self.norm(x))
        return residual + self.dropout(x)


class GridPulseTCN(nn.Module):
    def __init__(self, future_features=10, hidden=32):
        super().__init__()

        self.input_proj = nn.Conv1d(1, hidden, 1)

        self.tcn = nn.Sequential(
            TemporalBlock(hidden, dilation=1),
            TemporalBlock(hidden, dilation=2),
            TemporalBlock(hidden, dilation=4),
            TemporalBlock(hidden, dilation=8),
            TemporalBlock(hidden, dilation=16),
        )

        self.future_encoder = nn.Sequential(
            nn.Linear(future_features * 7, 32),
            nn.ReLU(),
        )

        self.head = nn.Sequential(
            nn.Linear(hidden + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 7)
        )

    def forward(self, history, future):
        x = history.unsqueeze(1)  # (batch, 1, 56)
        x = self.input_proj(x)
        x = self.tcn(x)

        history_features = x[:, :, -1]
        future_features = self.future_encoder(
            future.flatten(start_dim=1)
        )

        combined = torch.cat(
            [history_features, future_features],
            dim=1
        )

        return self.head(combined)


torch.manual_seed(42)

model = GridPulseTCN()

print(model)
print(
    "\nTrainable parameters:",
    sum(p.numel() for p in model.parameters())
)

GridPulseTCN(
  (input_proj): Conv1d(1, 32, kernel_size=(1,), stride=(1,))
  (tcn): Sequential(
    (0): TemporalBlock(
      (conv): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(2,))
      (norm): GroupNorm(4, 32, eps=1e-05, affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): TemporalBlock(
      (conv): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(2,))
      (norm): GroupNorm(4, 32, eps=1e-05, affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (2): TemporalBlock(
      (conv): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(8,), dilation=(4,))
      (norm): GroupNorm(4, 32, eps=1e-05, affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (3): TemporalBlock(
      (conv): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(16,), dilation=(8,))
      (norm): GroupNorm(4, 32, eps=1e-05, affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=Fal

Test one forward pass

In [8]:
history_tensor = torch.tensor(
    Xh_train_n[:4], dtype=torch.float32
)

future_tensor = torch.tensor(
    Xf_train_n[:4], dtype=torch.float32
)

model.eval()

with torch.no_grad():
    output = model(history_tensor, future_tensor)

print("Input history:", history_tensor.shape)
print("Input future:", future_tensor.shape)
print("Output:", output.shape)

assert output.shape == (4, 7)
assert torch.isfinite(output).all()

print("Forward pass successful.")

Input history: torch.Size([4, 56])
Input future: torch.Size([4, 7, 10])
Output: torch.Size([4, 7])
Forward pass successful.


Train the TCN

Prepare tensors and chronological split

In [9]:
from torch.utils.data import TensorDataset, DataLoader

torch.set_num_threads(8)
torch.manual_seed(42)

# Split using forecast origins, not randomly shuffled windows
fit_mask = kept_train < pd.Timestamp("2023-01-01")
stop_mask = kept_train >= pd.Timestamp("2023-01-01")

def make_loader(mask, shuffle):
    ds = TensorDataset(
        torch.from_numpy(Xh_train_n[mask]).float(),
        torch.from_numpy(Xf_train_n[mask]).float(),
        torch.from_numpy(y_train_n[mask]).float(),
    )
    return DataLoader(ds, batch_size=32, shuffle=shuffle)

fit_loader = make_loader(fit_mask, shuffle=True)
stop_loader = make_loader(stop_mask, shuffle=False)

print("Fitting windows:", fit_mask.sum())
print("Early-stopping windows:", stop_mask.sum())

Fitting windows: 282
Early-stopping windows: 358


Train with early stopping

In [10]:
import copy
import time

model = GridPulseTCN()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-3, weight_decay=1e-3
)
criterion = nn.HuberLoss(delta=0.5)

best_loss = float("inf")
best_state = None
patience = 20
stale = 0
history_log = []

start = time.perf_counter()

for epoch in range(1, 201):
    model.train()

    train_losses = []
    for xh, xf, y in fit_loader:
        optimizer.zero_grad(set_to_none=True)

        pred = model(xh, xf)
        loss = criterion(pred, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_losses.append(loss.item())

    model.eval()
    val_losses = []

    with torch.no_grad():
        for xh, xf, y in stop_loader:
            pred = model(xh, xf)
            val_losses.append(criterion(pred, y).item())

    train_loss = np.mean(train_losses)
    val_loss = np.mean(val_losses)

    history_log.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "stop_loss": val_loss,
    })

    if val_loss < best_loss - 1e-4:
        best_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        stale = 0
    else:
        stale += 1

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch:3d} | "
            f"train={train_loss:.4f} | "
            f"stop={val_loss:.4f}"
        )

    if stale >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)
model.eval()

print(f"\nBest epoch: {best_epoch}")
print(f"Best stopping loss: {best_loss:.4f}")
print(f"Training time: {(time.perf_counter()-start):.1f}s")

Epoch  10 | train=0.0702 | stop=0.0549
Epoch  20 | train=0.0519 | stop=0.0589
Epoch  30 | train=0.0394 | stop=0.0591
Early stopping at epoch 36

Best epoch: 16
Best stopping loss: 0.0487
Training time: 3.4s


Generate seven-day predictions

In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

model.eval()

with torch.inference_mode():
    predictions_normalized = model(
        torch.from_numpy(Xh_val_n).float(),
        torch.from_numpy(Xf_val_n).float()
    ).cpu().numpy()

# Convert normalized predictions back into MU
predictions_mu = (
    predictions_normalized * demand_std + demand_mean
)

assert predictions_mu.shape == (360, 7)
assert np.isfinite(predictions_mu).all()

tcn_mae = mean_absolute_error(
    y_val.ravel(),
    predictions_mu.ravel()
)

print(f"TCN 2024 MAE: {tcn_mae:.4f} MU")

TCN 2024 MAE: 7.2733 MU


In [12]:
tcn_results = pd.DataFrame({
    "origin": np.repeat(kept_val, 7),
    "horizon": np.tile(np.arange(1, 8), len(kept_val)),
    "tcn_pred": predictions_mu.ravel()
})

comparison = benchmark.merge(
    tcn_results,
    on=["origin", "horizon"],
    how="inner",
    validate="one_to_one"
)

assert len(comparison) == 2520

for name in [
    "chronos2_lora_calendar_200",
    "tcn_pred"
]:
    mae = mean_absolute_error(
        comparison["actual_mu"],
        comparison[name]
    )

    rmse = np.sqrt(mean_squared_error(
        comparison["actual_mu"],
        comparison[name]
    ))

    print(f"{name:30s} MAE={mae:.4f}  RMSE={rmse:.4f}")

chronos2_lora_calendar_200     MAE=6.8693  RMSE=9.5166
tcn_pred                       MAE=7.2733  RMSE=9.4830


Ensemble evaluation

In [13]:
comparison["ensemble_50_50"] = (
    0.5 * comparison["chronos2_lora_calendar_200"]
    + 0.5 * comparison["tcn_pred"]
)

models = [
    "chronos2_lora_calendar_200",
    "tcn_pred",
    "ensemble_50_50",
]

results = []

for name in models:
    errors = comparison[name] - comparison["actual_mu"]

    results.append({
        "model": name,
        "MAE": errors.abs().mean(),
        "RMSE": np.sqrt((errors**2).mean()),
        "Bias": errors.mean(),
    })

display(
    pd.DataFrame(results)
    .sort_values("MAE")
    .round(4)
)

,model,MAE,RMSE,Bias
2,ensemble_50_50,6.0547,8.1190,-1.8231
0,chronos2_lora_calendar_200,6.8693,9.5166,1.1044
1,tcn_pred,7.2733,9.4830,-4.7506


In [14]:
import torch
from pathlib import Path

MODEL_DIR = Path("../models/gridpulse_ensemble_v1")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "architecture": {
        "future_features": 10,
        "hidden": 32,
        "history_length": 56,
        "prediction_length": 7,
    },
    "normalization": {
        "demand_mean": float(demand_mean),
        "demand_std": float(demand_std),
        "temp_mean": temp_mean.tolist(),
        "temp_std": temp_std.tolist(),
    },
    "training": {
        "best_epoch": best_epoch,
        "best_stopping_loss": float(best_loss),
        "seed": 42,
    },
}, MODEL_DIR / "tcn_checkpoint.pt")

print("TCN saved successfully.")

TCN saved successfully.


In [15]:
import json

comparison.to_parquet(
    MODEL_DIR / "validation_2024_predictions.parquet",
    index=False
)

metadata = {
    "project": "GridPulse",
    "version": "ensemble_v1",
    "models": [
        "Chronos-2 calendar LoRA",
        "TCN trained from scratch"
    ],
    "ensemble_weights": {
        "chronos2": 0.5,
        "tcn": 0.5
    },
    "history_length": 56,
    "prediction_length": 7,
    "validation_year": 2024,
    "validation_origins": 360,
    "validation_mae_mu": 6.0547,
    "validation_rmse_mu": 8.1190,
    "weather_source": "Open-Meteo GFS Previous Runs",
    "weather_limitation": (
        "Fixed-lead historical forecasts; "
        "exact operational availability not verified."
    ),
}

(MODEL_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

print("Saved files:")
for path in MODEL_DIR.iterdir():
    print(path.name, f"{path.stat().st_size / 1024**2:.2f} MB")

# Verify checkpoint can be loaded
checkpoint = torch.load(
    MODEL_DIR / "tcn_checkpoint.pt",
    map_location="cpu",
    weights_only=True
)

restored = GridPulseTCN(
    future_features=checkpoint["architecture"]["future_features"],
    hidden=checkpoint["architecture"]["hidden"],
)

restored.load_state_dict(checkpoint["model_state_dict"])
restored.eval()

with torch.inference_mode():
    test_pred = restored(
        torch.from_numpy(Xh_val_n[:1]).float(),
        torch.from_numpy(Xf_val_n[:1]).float()
    )

print("\nCheckpoint restored successfully.")
print("Output shape:", tuple(test_pred.shape))

Saved files:
metadata.json 0.00 MB
tcn_checkpoint.pt 0.10 MB
validation_2024_predictions.parquet 0.10 MB

Checkpoint restored successfully.
Output shape: (1, 7)
